# A3C for Kung Fu

## Part 0 - Installing the required packages and importing the libraries

### Installing Gymnasium

In [1]:
!pip install gymnasium
!pip install "gymnasium[atari, accept-rom-license]"
!apt-get install -y swig
!pip install gymnasium[box2d]

"apt-get" non � riconosciuto come comando interno o esterno,
 un programma eseguibile o un file batch.


### Importing the libraries

In [2]:
import cv2
import math
import random
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torch.multiprocessing as mp
import torch.distributions as distributions
from torch.distributions import Categorical
import gymnasium as gym
from gymnasium import ObservationWrapper
from gymnasium.spaces import Box

## Part 1 - Building the AI

### Creating the architecture of the Neural Network

In [3]:
class Network(nn.Module):
    def __init__(self, action_size):
        super(Network, self).__init__()
        self.conv1 = torch.nn.Conv2d(in_channels=4,out_channels=32, kernel_size=(3,3), stride=2)
        self.conv2 = torch.nn.Conv2d(in_channels=32,out_channels=32, kernel_size=(3,3), stride=2)
        self.conv3 = torch.nn.Conv2d(in_channels=32,out_channels=32, kernel_size=(3,3), stride=2)
        self.flatten = torch.nn.Flatten()
        self.fc1 = torch.nn.Linear(32*9*9, 128)
        self.fc2a = torch.nn.Linear(128, action_size)
        self.fc2s = torch.nn.Linear(128, 1)
        
    def forward(self, state):
        x = self.conv1(state)
        x = F.relu(x)
        x = self.conv2(x)
        x = F.relu(x)
        x = self.conv3(x)
        x = F.relu(x)
        x = self.flatten(x)
        x = self.fc1(x)
        x = F.relu(x)
        action_value = self.fc2a(x)
        state_value = self.fc2s(x).squeeze(-1)
        return action_value, state_value

## Part 2 - Training the AI

### Setting up the environment

In [4]:
import ale_py
gym.register_envs(ale_py)

class PreprocessAtari(ObservationWrapper):

  def __init__(self, env, height = 42, width = 42, crop = lambda img: img, dim_order = 'pytorch', color = False, n_frames = 4):
    super(PreprocessAtari, self).__init__(env)
    self.img_size = (height, width)
    self.crop = crop
    self.dim_order = dim_order
    self.color = color
    self.frame_stack = n_frames
    n_channels = 3 * n_frames if color else n_frames
    obs_shape = {'tensorflow': (height, width, n_channels), 'pytorch': (n_channels, height, width)}[dim_order]
    self.observation_space = Box(0.0, 1.0, obs_shape)
    self.frames = np.zeros(obs_shape, dtype = np.float32)

  def reset(self):
    self.frames = np.zeros_like(self.frames)
    obs, info = self.env.reset()
    self.update_buffer(obs)
    return self.frames, info

  def observation(self, img):
    img = self.crop(img)
    img = cv2.resize(img, self.img_size)
    if not self.color:
      if len(img.shape) == 3 and img.shape[2] == 3:
        img = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
    img = img.astype('float32') / 255.
    if self.color:
      self.frames = np.roll(self.frames, shift = -3, axis = 0)
    else:
      self.frames = np.roll(self.frames, shift = -1, axis = 0)
    if self.color:
      self.frames[-3:] = img
    else:
      self.frames[-1] = img
    return self.frames

  def update_buffer(self, obs):
    self.frames = self.observation(obs)

def make_env():
  env = gym.make("KungFuMaster-v4", render_mode = 'rgb_array')
  env = PreprocessAtari(env, height = 84, width = 84, crop = lambda img: img, dim_order = 'pytorch', color = False, n_frames = 4)
  return env

env = make_env()

state_shape = env.observation_space.shape
number_actions = env.action_space.n
print("State shape:", state_shape)
print("Number actions:", number_actions)
print("Action names:", env.unwrapped.get_action_meanings())

State shape: (4, 84, 84)
Number actions: 14
Action names: ['NOOP', 'UP', 'RIGHT', 'LEFT', 'DOWN', 'DOWNRIGHT', 'DOWNLEFT', 'RIGHTFIRE', 'LEFTFIRE', 'DOWNFIRE', 'UPRIGHTFIRE', 'UPLEFTFIRE', 'DOWNRIGHTFIRE', 'DOWNLEFTFIRE']


### Initializing the hyperparameters

In [5]:
learning_rate = 1e-4
discount_factor = 0.99
number_environments = 16
n_steps = 5

### Implementing the A3C class

In [6]:
class Agent():
    
    def __init__(self, action_size):
        self.device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
        self.action_size = action_size
        self.network = Network(action_size).to(self.device)
        self.optimizer = optim.Adam(self.network.parameters(), lr = learning_rate)
    
    def act(self, state):
        if state.ndim == 3:
            state = [state]
        state = torch.tensor(state, dtype = torch.float32, device = self.device)
        action_values, _ = self.network(state)
        policy = F.softmax(action_values, dim=-1)
        return np.array([np.random.choice(len(p), p=p) for p in policy.detach().cpu().numpy()])

    def step(self, states_buf, actions_buf, rewards_buf, dones_buf, last_states):
        last_states_t = torch.tensor(np.array(last_states), dtype=torch.float32, device=self.device)
        with torch.no_grad():
            _, last_values = self.network(last_states_t)
        returns = last_values
        all_returns = []
        for t in reversed(range(len(rewards_buf))):
            rewards = torch.tensor(rewards_buf[t], dtype=torch.float32, device=self.device)
            dones = torch.tensor(dones_buf[t], dtype=torch.bool, device=self.device).float()
            returns = rewards + discount_factor * returns * (1 - dones)
            all_returns.insert(0, returns)
        all_states = torch.tensor(np.concatenate(states_buf, axis=0), dtype=torch.float32, device=self.device)
        all_actions = torch.tensor(np.concatenate(actions_buf, axis=0), dtype=torch.long, device=self.device)
        all_returns = torch.cat(all_returns, dim=0).detach()
        action_logits, state_values = self.network(all_states)
        advantage = all_returns - state_values
        probs = F.softmax(action_logits, dim=-1)
        log_probs = F.log_softmax(action_logits, dim=-1)
        entropy = -torch.sum(probs * log_probs, dim=-1)
        n = all_states.shape[0]
        logp_actions = log_probs[torch.arange(n, device=self.device), all_actions]
        actor_loss = -torch.mean(logp_actions * advantage.detach()) - 0.001 * torch.mean(entropy)
        critic_loss = F.mse_loss(state_values, all_returns)
        total_loss = actor_loss + critic_loss
        self.optimizer.zero_grad()
        total_loss.backward()
        torch.nn.utils.clip_grad_norm_(self.network.parameters(), 0.5)
        self.optimizer.step()

### Initializing the A3C agent

In [7]:
agent = Agent(number_actions)

### Evaluating our A3C agent on a certain number of episodes

In [8]:
def evaluate(agent, env, n_episodes = 1):
    episode_rewards = []
    for _ in range(n_episodes):
        state, _ = env.reset()
        total_reward = 0
        while True:
            action = agent.act(state)[0]
            state, reward, terminated, truncated, _ = env.step(action)
            total_reward += reward
            if terminated or truncated:
                break
        episode_rewards.append(total_reward)
    return episode_rewards

### Managing multiple environments simultaneously

In [9]:
class EnvBatch:
    def __init__(self, n_envs=10):
        self.envs = [make_env() for _ in range(n_envs)]
    
    def reset(self):
        _states = []
        for env in self.envs:
            state, _ = env.reset()
            _states.append(state)
        return np.array(_states)
    
    def step(self, actions):
        results = [env.step(a) for env, a in zip(self.envs, actions)]
        next_states = np.array([r[0] for r in results])
        rewards = np.array([r[1] for r in results])
        dones = np.array([r[2] or r[3] for r in results])
        for i in range(len(self.envs)):
            if dones[i]:
                next_states[i] = self.envs[i].reset()[0]
        return next_states, rewards, dones

### Training the A3C agent

In [10]:
import tqdm

env_batch = EnvBatch(number_environments)
batch_states = env_batch.reset()

with tqdm.trange(0, 10001) as progress_bar:
  for i in progress_bar:
    states_buf, actions_buf, rewards_buf, dones_buf = [], [], [], []
    for _ in range(n_steps):
      batch_actions = agent.act(batch_states)
      batch_next_states, batch_rewards, batch_dones = env_batch.step(batch_actions)
      states_buf.append(batch_states)
      actions_buf.append(batch_actions)
      rewards_buf.append(batch_rewards * 0.01)
      dones_buf.append(batch_dones)
      batch_states = batch_next_states
    agent.step(states_buf, actions_buf, rewards_buf, dones_buf, batch_states)
    if i % 1000 == 0:
      print("Average agent reward: ", np.mean(evaluate(agent, env, n_episodes = 10)))

  0%|          | 0/10001 [00:00<?, ?it/s]C:\Users\gabri\AppData\Local\Temp\ipykernel_15952\3353567776.py:12: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\torch\csrc\utils\tensor_new.cpp:281.)
  state = torch.tensor(state, dtype = torch.float32, device = self.device)
  0%|          | 2/10001 [01:54<131:26:12, 47.32s/it] 

Average agent reward:  400.0


 10%|█         | 1003/10001 [04:42<23:03:25,  9.22s/it]

Average agent reward:  3320.0


 20%|██        | 2003/10001 [07:16<26:21:35, 11.86s/it]

...

Average agent reward:  11950.0


 90%|█████████ | 9003/10001 [31:58<4:58:42, 17.96s/it] 

Average agent reward:  13230.0


100%|██████████| 10001/10001 [35:22<00:00,  4.71it/s] 

Average agent reward:  13420.0


## Part 3 - Visualizing the results

In [11]:
import glob
import io
import base64
import imageio
from IPython.display import HTML, display

def show_video_of_model(agent):
    env = gym.make("KungFuMaster-v4", render_mode='rgb_array')
    env = PreprocessAtari(env, height=84, width=84, crop=lambda img: img, dim_order='pytorch', color=False, n_frames=4)
    state, _ = env.reset()
    done = False
    frames = []
    while not done:
        frame = env.render()
        frames.append(frame)
        action = agent.act(state)[0]
        state, reward, terminated, truncated, _ = env.step(action)
        done = terminated or truncated
    env.close()
    imageio.mimsave('video.mp4', frames, fps=30)

show_video_of_model(agent)

def show_video():
    mp4list = glob.glob('*.mp4')
    if len(mp4list) > 0:
        mp4 = mp4list[0]
        video = io.open(mp4, 'r+b').read()
        encoded = base64.b64encode(video)
        display(HTML(data='''<video alt="test" autoplay
                loop controls style="height: 400px;">
                <source src="data:video/mp4;base64,{0}" type="video/mp4" />
             </video>'''.format(encoded.decode('ascii'))))
    else:
        print("Could not find video")

show_video()

IMAGEIO FFMPEG_WRITER WARNING: input image is not divisible by macro_block_size=16, resizing from (160, 210) to (160, 224) to ensure video compatibility with most codecs and players. To prevent resizing, make your input image divisible by the macro_block_size or set the macro_block_size to 1 (risking incompatibility).
